🌳 Agroforestry Plot Simulator

This script generates a **typical agroforestry plot** for coffee, cacao, or banana systems, with randomized or user-specified species composition, shading contribution, and a realistic geographic location in the **Dominican Republic**, consistent with the crop’s ecological requirements (including elevation).

It uses the [Open-Elevation API](https://open-elevation.com/) to ensure the sampled plot coordinates fall within the appropriate elevation range for the selected crop.

---

## ✨ Features

✅ Simulates an agroforestry plot for:
- ☕ Coffee (*Coffea arabica*) — upper watershed, mountains
- 🍫 Cacao (*Theobroma cacao*) — mid-elevation foothills
- 🍌 Banana (*Musa spp.*) — lowland plains

✅ Outputs a `pandas.DataFrame` listing:
- Species (main crop & associated shade trees)
- Scientific name
- Number of plants per hectare
- Per-tree shading contribution (%)
- Plot size
- Yield estimate (if available)
- Latitude, Longitude, Elevation (validated)

✅ Ensures:
- Coordinates fall within appropriate **regions of the Dominican Republic**
- Elevation is within the crop’s acceptable range
- Species proportions & shading are randomized within realistic ranges

---

## 🔧 User Input Options

You can let the script randomly generate all parameters, or **provide specific values** where available:  
✅ Provide exact **latitude & longitude**  
✅ Provide exact **number of trees per species** (`user_species_counts`)  
✅ Provide exact **per-tree shading values per species** (`user_species_shading`)  

If any of these are omitted, the script will generate random plausible values.

---


In [35]:
import pandas as pd
import numpy as np
import requests
import time

# bounding boxes for each system
region_bounds = {
    "Coffee": {"lat": (18.8, 19.2), "lon": (-70.7, -70.5)},
    "Cacao":  {"lat": (19.2, 19.4), "lon": (-70.3, -70.0)},
    "Banana": {"lat": (19.5, 19.9), "lon": (-71.7, -70.9)}
}

# elevation ranges for each system (m)
elevation_ranges = {
    "Coffee": (600, 1500),
    "Cacao":  (100, 400),
    "Banana": (0, 100)
}

species_data = {
    "Coffee": [
        {"Species": "Guama", "Scientific name": "Inga spp.", "Shade range": (30, 60)},
        {"Species": "Bitter orange", "Scientific name": "Citrus aurantium", "Shade range": (30, 50)},
        {"Species": "Sweet orange", "Scientific name": "Citrus sinensis", "Shade range": (30, 50)},
        {"Species": "Sapote", "Scientific name": "Pouteria sapota", "Shade range": (30, 50)},
        {"Species": "Breadfruit", "Scientific name": "Artocarpus altilis", "Shade range": (30, 50)},
        {"Species": "Avocado", "Scientific name": "Persea americana", "Shade range": (30, 50)},
    ],
    "Cacao": [
        {"Species": "Cuban cedar", "Scientific name": "Gliricidia sepium", "Shade range": (40, 60)},
        {"Species": "Bitter orange", "Scientific name": "Citrus aurantium", "Shade range": (30, 50)},
        {"Species": "Sweet orange", "Scientific name": "Citrus sinensis", "Shade range": (30, 50)},
        {"Species": "Sapote", "Scientific name": "Pouteria sapota", "Shade range": (30, 50)},
        {"Species": "Breadfruit", "Scientific name": "Artocarpus altilis", "Shade range": (30, 50)},
        {"Species": "Avocado", "Scientific name": "Persea americana", "Shade range": (30, 50)},
    ],
    "Banana": [
        {"Species": "Banana (main crop)", "Scientific name": "Musa spp.", "Shade range": None},
        {"Species": "Coconut", "Scientific name": None, "Shade range": None},
        {"Species": "Cacao", "Scientific name": None, "Shade range": None},
        {"Species": "Citrus", "Scientific name": None, "Shade range": None},
        {"Species": "Soursop", "Scientific name": None, "Shade range": None},
        {"Species": "Lipia", "Scientific name": None, "Shade range": None},
    ]
}

system_info = {
    "Coffee": {"plot_size_ha": 1.0, "total_shade_trees": 144, "yield_ton_per_ha": 0.73},
    "Cacao":  {"plot_size_ha": 1.0, "total_shade_trees": 144, "yield_ton_per_ha": 0.50},
    "Banana": {"plot_size_ha": 2.5, "total_shade_trees": 0,   "yield_ton_per_ha": 22.64},
}

main_crop_scientific = {
    "Coffee": "Coffea arabica",
    "Cacao": "Theobroma cacao",
    "Banana": "Musa spp."
}

def get_elevation(lat, lon):
    url = "https://api.open-elevation.com/api/v1/lookup"
    params = {"locations": f"{lat},{lon}"}
    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200:
            result = response.json()
            elevation = result['results'][0]['elevation']
            return elevation
        else:
            print(f"Error: Status {response.status_code}")
            return None
    except Exception as e:
        print(f"Error fetching elevation: {e}")
        return None

def sample_valid_location(system, max_attempts=20, sleep_sec=0.5):
    min_elev, max_elev = elevation_ranges[system]
    for attempt in range(1, max_attempts + 1):
        lat = np.random.uniform(*region_bounds[system]["lat"])
        lon = np.random.uniform(*region_bounds[system]["lon"])
        elev = get_elevation(lat, lon)
        if elev is None:
            print(f"Attempt {attempt}: Failed to fetch elevation.")
            time.sleep(sleep_sec)
            continue
        if min_elev <= elev <= max_elev:
            print(f"✅ Found valid location at attempt {attempt}: lat={lat:.4f}, lon={lon:.4f}, elev={elev}m")
            return lat, lon, elev
        else:
            print(f"Attempt {attempt}: Elevation {elev}m outside range [{min_elev}, {max_elev}]")
            time.sleep(sleep_sec)
    raise RuntimeError(f"Could not find valid location for {system} in {max_attempts} attempts.")

def generate_typical_plot_with_elevation(
    system: str,
    seed: int = None,
    user_lat: float = None,
    user_lon: float = None,
    user_species_counts: dict = None,
    user_species_shading: dict = None
) -> pd.DataFrame:
    np.random.seed(seed)
    info = system_info[system]
    species_list = species_data[system]
    rows = []

    # lat/lon/elevation
    if user_lat is not None and user_lon is not None:
        lat, lon = user_lat, user_lon
        elev = get_elevation(lat, lon)
        if elev is None:
            raise ValueError("Failed to fetch elevation for given coordinates.")
    else:
        lat, lon, elev = sample_valid_location(system)

    # main crop
    rows.append({
        "System": system,
        "Plot size (ha)": info["plot_size_ha"],
        "Species": f"{system} (main crop)",
        "Scientific name": main_crop_scientific[system],
        "Plants/ha": None,
        "Yield (t/ha/year)": info["yield_ton_per_ha"],
        "Per-tree shading (%)": None,
        "Latitude": lat,
        "Longitude": lon,
        "Elevation (m)": elev
    })

    if info["total_shade_trees"] > 0:
        total_shade_trees = info["total_shade_trees"]
        if user_species_counts:
            counts = [user_species_counts.get(sp["Species"], 0) for sp in species_list]
        else:
            proportions = np.random.dirichlet(np.ones(len(species_list)))
            counts = np.round(proportions * total_shade_trees).astype(int)

        for sp, count in zip(species_list, counts):
            if user_species_shading and sp["Species"] in user_species_shading:
                shade_val = user_species_shading[sp["Species"]]
            elif sp["Shade range"]:
                shade_val = np.random.uniform(*sp["Shade range"])
            else:
                shade_val = None

            rows.append({
                "System": system,
                "Plot size (ha)": info["plot_size_ha"],
                "Species": sp["Species"],
                "Scientific name": sp["Scientific name"],
                "Plants/ha": count,
                "Yield (t/ha/year)": None,
                "Per-tree shading (%)": round(shade_val, 1) if shade_val else None,
                "Latitude": lat,
                "Longitude": lon,
                "Elevation (m)": elev
            })
    else:
        for sp in species_list:
            rows.append({
                "System": system,
                "Plot size (ha)": info["plot_size_ha"],
                "Species": sp["Species"],
                "Scientific name": sp["Scientific name"],
                "Plants/ha": None,
                "Yield (t/ha/year)": info["yield_ton_per_ha"] if "Banana" in sp["Species"] else None,
                "Per-tree shading (%)": None,
                "Latitude": lat,
                "Longitude": lon,
                "Elevation (m)": elev
            })

    return pd.DataFrame(rows)

# Example:
df = generate_typical_plot_with_elevation("Coffee")
print(df)


✅ Found valid location at attempt 1: lat=18.9857, lon=-70.6193, elev=1154.0m
   System  Plot size (ha)             Species     Scientific name  Plants/ha  \
0  Coffee             1.0  Coffee (main crop)      Coffea arabica        NaN   
1  Coffee             1.0               Guama           Inga spp.       15.0   
2  Coffee             1.0       Bitter orange    Citrus aurantium       19.0   
3  Coffee             1.0        Sweet orange     Citrus sinensis       14.0   
4  Coffee             1.0              Sapote     Pouteria sapota        5.0   
5  Coffee             1.0          Breadfruit  Artocarpus altilis       30.0   
6  Coffee             1.0             Avocado    Persea americana       61.0   

   Yield (t/ha/year)  Per-tree shading (%)   Latitude  Longitude  \
0               0.73                   NaN  18.985674  -70.61927   
1                NaN                  58.8  18.985674  -70.61927   
2                NaN                  49.7  18.985674  -70.61927   
3         

In [32]:
df

,System,Plot size (ha),Species,Scientific name,Plants/ha,Yield (t/ha/year),Per-tree shading (%),Latitude,Longitude,Elevation (m)
0,Coffee,1.0,Coffee (main crop),Coffea arabica,NaN,0.73,NaN,18.949816,-70.509857,1221.0
1,Coffee,1.0,Guama,Inga spp.,41.0,NaN,48.0,18.949816,-70.509857,1221.0
2,Coffee,1.0,Bitter orange,Citrus aurantium,28.0,NaN,44.2,18.949816,-70.509857,1221.0
3,Coffee,1.0,Sweet orange,Citrus sinensis,5.0,NaN,30.4,18.949816,-70.509857,1221.0
4,Coffee,1.0,Sapote,Pouteria sapota,5.0,NaN,49.4,18.949816,-70.509857,1221.0
5,Coffee,1.0,Breadfruit,Artocarpus altilis,2.0,NaN,46.6,18.949816,-70.509857,1221.0
6,Coffee,1.0,Avocado,Persea americana,62.0,NaN,34.2,18.949816,-70.509857,1221.0


In [36]:
df

,System,Plot size (ha),Species,Scientific name,Plants/ha,Yield (t/ha/year),Per-tree shading (%),Latitude,Longitude,Elevation (m)
0,Coffee,1.0,Coffee (main crop),Coffea arabica,NaN,0.73,NaN,18.985674,-70.61927,1154.0
1,Coffee,1.0,Guama,Inga spp.,15.0,NaN,58.8,18.985674,-70.61927,1154.0
2,Coffee,1.0,Bitter orange,Citrus aurantium,19.0,NaN,49.7,18.985674,-70.61927,1154.0
3,Coffee,1.0,Sweet orange,Citrus sinensis,14.0,NaN,30.6,18.985674,-70.61927,1154.0
4,Coffee,1.0,Sapote,Pouteria sapota,5.0,NaN,47.6,18.985674,-70.61927,1154.0
5,Coffee,1.0,Breadfruit,Artocarpus altilis,30.0,NaN,38.0,18.985674,-70.61927,1154.0
6,Coffee,1.0,Avocado,Persea americana,61.0,NaN,44.5,18.985674,-70.61927,1154.0
